# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdeenMir/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane: CTR / Engagement Opportunity Scoring** (locked week 4). Week-4 baseline: pages whose
CTR sits below their position tier's expected CTR, filtered by a volume floor.

**Target for this week:** `is_declining_label` (`trend_direction == "down"`), the same
lane-agnostic proxy outcome this repo's own starter pipeline uses to score baseline vs model
(`docs/ml-intern-dataset-and-lane-guide.md`, section 5 -- base rate 54.2%). I am **not** using
my own CTR-gap rule as the target: that would be circular -- a model trained to reproduce a
threshold I hand-picked "discovers" nothing, it just re-learns my cutoff. `is_declining_label`
is an independent, already-observed outcome, so beating it is a real test.

**Methods, from the week's menu:**
- **Logistic Regression** first -- readable, coefficients have a sign I can sanity-check
  (does higher CTR really point away from decline?).
- **Decision Tree** (depth-limited) -- printable, gives a plain-language rule to compare
  directly against my hand-written baseline rule.
- **Random Forest** -- the "stronger" step once the readable models set a floor; only worth
  keeping if it actually beats the simpler models on the same metric.

Simplicity is the default; Random Forest only earns its place in section 3 if the numbers say so.

**Leakage guard:** `trend_direction` / `trend_pct` are the label -- never features. Their
direct inputs, `impressions_last_30d` / `impressions_prev_30d` / `clicks_last_30d` /
`clicks_prev_30d` / `sessions_last_30d` / `sessions_prev_30d`, are excluded too: a model could
reconstruct `trend_pct` almost exactly from those two windows, which is leakage by another
name. Features come from the 90-day totals and derived rates only -- the same feature set the
starter pipeline documents (`scripts/ml_utils.py: MODEL_NUMERIC_FEATURES` /
`MODEL_CATEGORICAL_FEATURES`), plus explicit missingness flags (see below) instead of a blind
`fillna(0)`, since missingness follows `content_type` (data dictionary warning).

In [1]:
import os
import numpy as np
import pandas as pd

LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"
RAW_URL = "https://raw.githubusercontent.com/AdeenMir/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
RUNNING_LOCAL = os.path.exists(LOCAL_PATH)
OUTPUT_DIR = "../outputs" if RUNNING_LOCAL else "work/outputs"

df = pd.read_csv(LOCAL_PATH if RUNNING_LOCAL else RAW_URL)
print("rows:", len(df), "| environment:", "local clone" if RUNNING_LOCAL else "Colab / no local clone")

# --- Target: independent of my own baseline rule ---
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("\nBase rate (is_declining_label):", df["is_declining_label"].mean().round(4))

# --- Missingness flags before imputing (missingness follows content_type) ---
for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
    df[f"has_{col}"] = df[col].notna().astype(int)

# --- Feature set: same list the starter pipeline documents, minus anything that touches
#     the label window (impressions/clicks/sessions last_30d / prev_30d are excluded on
#     purpose -- see leakage guard above) ---
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "has_search_volume", "has_competition", "has_cpc", "has_word_count", "has_char_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("unknown").astype(str)

X = pd.get_dummies(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], columns=CATEGORICAL_FEATURES, drop_first=True)
y = df["is_declining_label"]
print("\nfeature matrix shape:", X.shape)
print("label columns excluded from X:", "trend_direction" in X.columns, "trend_pct" in X.columns)


rows: 30000 | environment: Colab / no local clone

Base rate (is_declining_label): 0.5421

feature matrix shape: (30000, 49)
label columns excluded from X: False False


## 2. Split design

**Grouped by `client_id`, 70/30, `GroupShuffleSplit(random_state=42)`.**

Rows from the same client aren't independent -- they share a site, a template, a CMS, an SEO
team, seasonality. A random row-level split would let near-duplicate pages from the same
client land in both train and test, so the model could partly memorize a client instead of
learning a general pattern -- w03's data contract flagged the same risk for grouped splits.
Grouping by `client_id` keeps every client entirely on one side of the split, which is the
honest test of whether this generalizes to a client the model has never seen.

Not time-aware: the starter CSV is a single trailing-90-day snapshot per page, not a
longitudinal panel, so there's no earlier/later window to split on here (that's what the
warehouse's `fact_content_daily_performance` table is for, in a later week).

In [2]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

print(f"train rows: {len(X_train)}  ({y_train.mean():.3f} positive rate)")
print(f"test rows:  {len(X_test)}  ({y_test.mean():.3f} positive rate)")

overlap = set(df_train['client_id']) & set(df_test['client_id'])
print(f"\nclients in both train and test: {len(overlap)} (must be 0)")
print(f"distinct clients -- train: {df_train['client_id'].nunique()}, test: {df_test['client_id'].nunique()}")


train rows: 19166  (0.532 positive rate)
test rows:  10834  (0.559 positive rate)

clients in both train and test: 0 (must be 0)
distinct clients -- train: 22, test: 10


## 3. Train + compare vs my baseline

Same test rows, same metric, for all four rows of the table below: ROC AUC, average
precision, and precision@50 against `is_declining_label`, plus the base rate for scale.

**Baseline, refit fairly:** week 4 scored the whole dataset using tier-median CTR computed
from the whole dataset. Scoring it against a target here on the *same test split* the models
use would be an unfair edge for the baseline if its own "expected CTR by tier" table peeked at
test rows. So the tier-expected-CTR table below is refit on **train only**, then applied to
test -- same rule, same volume floor (500 impressions_90d), fit the same way a model would be.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
VOLUME_FLOOR = 500  # same floor as week 4, justified there by the CTR-noise signal check

# ---- Baseline: week-4 rule, tier-expected-CTR table fit on TRAIN only ----
train_reliable = df_train[(df_train["avg_position"] > 0) & (df_train["impressions_90d"] >= VOLUME_FLOOR)]
expected_ctr_by_tier = train_reliable.groupby("position_tier")["ctr"].median()

def baseline_score(frame):
    has_pos = frame["avg_position"] > 0
    expected = frame["position_tier"].map(expected_ctr_by_tier)
    gap = (expected - frame["ctr"]).clip(lower=0).where(has_pos, 0)
    meets_floor = has_pos & (frame["impressions_90d"] >= VOLUME_FLOOR)

    def normalize(s):
        lo, hi = s.min(), s.max()
        return (s - lo) / (hi - lo) if hi > lo else s * 0
    def pct_rank(s):
        return s.rank(method="average", pct=True).fillna(0)

    score = normalize(gap) * pct_rank(frame["impressions_90d"])
    return np.where(meets_floor, score, 0.0)

baseline_test_score = baseline_score(df_test)

# ---- Models, trained on train split only ----
# Logistic regression needs scaled inputs to converge cleanly (unscaled dummy + raw-count
# features mix wildly different ranges); trees are scale-invariant, so they use raw X.
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE).fit(X_train_scaled, y_train)
tree = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

scores = {
    "baseline rule (week 4, refit on train)": baseline_test_score,
    "logistic regression": log_reg.predict_proba(X_test_scaled)[:, 1],
    "decision tree (depth=4)": tree.predict_proba(X_test)[:, 1],
    "random forest": forest.predict_proba(X_test)[:, 1],
}

def precision_at_k(y_true, score, k):
    order = np.argsort(-np.asarray(score))
    return float(np.asarray(y_true)[order[:k]].mean())

rows = []
for name, s in scores.items():
    rows.append({
        "method": name,
        "roc_auc": round(roc_auc_score(y_test, s), 3),
        "average_precision": round(average_precision_score(y_test, s), 3),
        "precision_at_50": round(precision_at_k(y_test, s, 50), 3),
    })

comparison = pd.DataFrame(rows)
base_rate = round(y_test.mean(), 3)
print(f"Test-set base rate (is_declining_label): {base_rate}\n")
print(comparison.to_string(index=False))


Test-set base rate (is_declining_label): 0.559

                                method  roc_auc  average_precision  precision_at_50
baseline rule (week 4, refit on train)    0.562              0.604             0.72
                   logistic regression    0.620              0.653             0.70
               decision tree (depth=4)    0.582              0.609             0.68
                         random forest    0.623              0.656             0.76


## 4. Errors and interpretation

**What the table says, plainly:** random forest wins on all three metrics and earns its
complexity. Logistic regression beats the baseline on ROC AUC and average precision but not on
precision@50 -- the metric that matters most for a reviewer working top-down. The depth-4
decision tree is the clearest "complexity doesn't pay for itself" case: reasonable ROC AUC, but
its precision@50 is *worse* than the hand-written baseline. Cell below shows why: at depth 4
the tree only has a handful of leaves, so most of the 10,834 test rows share one of a dozen-odd
predicted probabilities -- ranking the top 50 among a sea of ties is close to arbitrary. A
deeper tree would fix that but stops being "a rule I can print and read," which is the whole
point of using a tree here.

**What the winning model leans on:** random forest's top features are `avg_position`,
`log_impressions_90d`, `days_with_impressions`, `content_age_days`, `log_sessions_90d` -- all
plausible (position and exposure are exactly what the CTR lane's own logic already treats as
central), and no single feature dominates the way a leaked column would (a leaked feature
usually shows up as one importance far above the rest). That's a reasonable pass on the
"suspiciously perfect" check.

**Three concrete wrong cases (random forest):**
- Two of the model's most confident false positives are pages with near-zero CTR at a
  reasonable position and moderate volume -- exactly the shape my week-4 rule would flag for a
  title/meta review. The model reads "structurally weak" and predicts decline; the actual
  label was `up` or `stable`. Plausible reading: a page can be under-capturing clicks (my
  lane's real question) without currently trending down (this week's borrowed label) -- they're
  related but not the same thing, and this is direct evidence of that gap.
- The clearest false negatives are pages with `impressions_90d` of 1 or 2 -- essentially no
  exposure data at all, correctly labeled `down` but the model (like a human) has almost
  nothing to reason from and predicts confidently the wrong way. This is a data-sparsity
  failure, not a feature-quality one, and it's exactly the operating range my week-4 baseline's
  volume floor was built to abstain from rather than guess at -- these rows would score 0 and
  get no action label under the baseline, which is the more honest behavior of the two.

In [4]:
# Tie diagnosis for the depth-4 tree (why precision@50 lags a simple baseline)
tree_proba = tree.predict_proba(X_test)[:, 1]
print(f"decision tree: {tree.get_n_leaves()} leaves, "
      f"{len(np.unique(tree_proba))} unique predicted probabilities across {len(X_test)} test rows")

# What the winning model (random forest) leans on
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 random forest feature importances:")
print(importances.head(10).round(4))

top_share = importances.iloc[0]
second_share = importances.iloc[1]
print(f"\nTop feature share: {top_share:.3f}, second: {second_share:.3f} "
      f"-- {'no single feature dominates (looks clean)' if top_share < 0.30 else 'CHECK: one feature dominates, possible leakage'}")

# Three concrete wrong cases
res = df_test.copy()
res["proba_rf"] = forest.predict_proba(X_test)[:, 1]
res["pred_rf"] = (res["proba_rf"] >= 0.5).astype(int)
res["true"] = y_test.values

cols = ["content_id", "avg_position", "ctr", "impressions_90d", "days_since_last_update",
        "word_count", "trend_direction", "proba_rf"]

false_positives = res[(res["pred_rf"] == 1) & (res["true"] == 0)].sort_values("proba_rf", ascending=False).head(3)
false_negatives = res[(res["pred_rf"] == 0) & (res["true"] == 1)].sort_values("proba_rf", ascending=True).head(3)

print("\nMost confident false positives (predicted declining, actually was not):")
print(false_positives[cols].to_string(index=False))
print("\nMost confident false negatives (predicted not declining, actually was):")
print(false_negatives[cols].to_string(index=False))

print(f"\nFalse-negative impressions_90d values, for scale: "
      f"{false_negatives['impressions_90d'].tolist()} (week-4 volume floor was {VOLUME_FLOOR})")


decision tree: 16 leaves, 14 unique predicted probabilities across 10834 test rows

Top 10 random forest feature importances:
avg_position             0.1096
log_impressions_90d      0.1057
days_with_impressions    0.0928
content_age_days         0.0757
log_sessions_90d         0.0521
word_count               0.0498
char_count               0.0483
days_with_sessions       0.0473
ctr                      0.0470
scroll_rate              0.0453
dtype: float64

Top feature share: 0.110, second: 0.106 -- no single feature dominates (looks clean)

Most confident false positives (predicted declining, actually was not):
          content_id  avg_position  ctr  impressions_90d  days_since_last_update  word_count trend_direction  proba_rf
content_2ba626fea4d6           7.2 0.00              360                     104      1405.0              up  0.976667
content_1d0963b56227          39.0 0.09             3445                     104      1480.0              up  0.966667
content_d015eb800625   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.